# Persistence & Checkpointing：失败后使用 PostgresSaver 恢复运行

> 适用版本：本项目锁定的 **LangGraph 1.1.2**、**langgraph-checkpoint-postgres 3.0.5** 与 **psycopg 3.3.3**。示例不调用大模型或外部 API；节点失败是教学代码主动制造的预期现象。

本笔记演示一条完整的故障恢复链路：

1. `prepare_job` 完成并形成完整 checkpoint；
2. 下一 superstep 中，`stable_branch` 成功，`flaky_branch` 抛出异常；
3. `PostgresSaver` 把成功分支的结果和失败信息保存为 pending writes；
4. 关闭第一个 saver，创建全新的 saver 与图，模拟应用重启后的新连接；
5. 使用相同 `thread_id` 调用 `graph.invoke(None, config)`；
6. LangGraph 复用成功分支的 pending writes，只重新执行失败分支，随后完成汇总。

~~~mermaid
flowchart LR
    A["第一次运行"] --> P["prepare_job 成功<br/>完整 checkpoint"]
    P --> S["stable_branch 成功<br/>pending write"]
    P --> F["flaky_branch 失败<br/>error write"]
    S --> DB[(PostgreSQL)]
    F --> DB
    DB --> X["关闭 saver / 修复故障"]
    X --> R["新 saver + 相同 thread_id<br/>invoke(None, config)"]
    R --> C["复用 stable_branch 结果"]
    R --> FR["只重跑 flaky_branch"]
    C --> J["finalize"]
    FR --> J
~~~

> 这里用“关闭旧 saver、打开新 saver”验证状态确实来自 PostgreSQL，同时保留当前 Notebook 的执行计数器，以证明哪些节点没有重跑。真正跨 Python 进程恢复时，图定义、State Schema、节点名称、数据库和 `thread_id` 必须保持兼容。


In [1]:
import operator
import os
from collections import Counter
from importlib.metadata import version
from typing import Annotated, TypedDict
from urllib.parse import urlsplit
from uuid import uuid4

import psycopg
from dotenv import load_dotenv
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph

print("LangGraph version:", version("langgraph"))
print(
    "Postgres checkpointer version:",
    version("langgraph-checkpoint-postgres"),
)
print("psycopg version:", version("psycopg"))


LangGraph version: 1.1.2
Postgres checkpointer version: 3.0.5
psycopg version: 3.3.3


## 1. 恢复机制：完整 checkpoint 与 pending writes

LangGraph 在每个 superstep 边界保存完整 checkpoint。若同一个 superstep 中有多个并行任务，其中一部分成功、一部分失败，则这一轮无法形成“全部任务都已完成”的下一个完整 checkpoint，但成功任务的节点级写入仍会通过 `put_writes()` 保存。

| 持久化内容 | 保存时机 | 恢复时的作用 |
| --- | --- | --- |
| 完整 checkpoint | superstep 边界 | 恢复已提交 State、下一批任务和时间线 |
| pending writes | 单个任务完成或失败时 | 复用已成功任务的输出，并记录失败任务 |
| `thread_id` | 每次调用的 `configurable` 配置 | 定位同一条 checkpoint 时间线 |

恢复异常中止的 Graph API 运行时，关键调用是：

~~~python
graph.invoke(None, same_config)
~~~

- `None` 表示继续未完成的运行，不是提交一轮新业务输入；
- 必须复用原来的 `thread_id`；换 ID 会得到一条全新的空时间线；
- 不要再次传入原始输入字典来“恢复”，那会被解释为新的输入，并可能丢弃当前未完成任务；
- `Command(resume=...)` 主要用于给 `interrupt()` 提供恢复值，本例是异常恢复，不需要它。

本例显式使用 `durability="sync"`，让完整 checkpoint 在进入下一 superstep 前同步完成持久化。它降低最近边界尚未落稳的窗口，但不承诺外部副作用 exactly-once；失败节点仍会从节点函数开头重新执行。


## 2. PostgreSQL 配置与安全探测

本 Notebook 沿用 `02-postgres-saver.ipynb` 的配置约定：

1. 优先读取 `LANGGRAPH_POSTGRES_URI`；
2. 其次兼容项目已有的 `LANGCHAIN_POSTGRES_URL`；
3. 都不存在时使用公开的本地教学占位符。

代码只打印变量名与状态，不输出 URI、密码、环境主机名或原始数据库异常。首次使用目标数据库/schema 时需要执行一次 `PostgresSaver.setup()`；重复调用会根据迁移记录补齐缺失版本。

若 PostgreSQL 当前不可用，所有数据库步骤都会明确输出 `SKIP`。此时只能确认代码、导入和结构有效，不能声称恢复链路已经连接数据库验证。


In [2]:
DEFAULT_POSTGRES_URI = (
    "postgresql://langgraph:langgraph-local-only"
    "@localhost:5432/langgraph?sslmode=disable"
)
# 沿用项目 Notebook 约定；不要读取或打印 .env 文件内容。
dotenv_loaded = load_dotenv(override=True)
langgraph_postgres_uri = os.getenv("LANGGRAPH_POSTGRES_URI")
existing_project_postgres_uri = os.getenv("LANGCHAIN_POSTGRES_URL")

if langgraph_postgres_uri:
    POSTGRES_URI = langgraph_postgres_uri
    postgres_uri_variable = "LANGGRAPH_POSTGRES_URI"
elif existing_project_postgres_uri:
    POSTGRES_URI = existing_project_postgres_uri
    postgres_uri_variable = "LANGCHAIN_POSTGRES_URL"
else:
    POSTGRES_URI = DEFAULT_POSTGRES_URI
    postgres_uri_variable = None

POSTGRES_CONNECT_TIMEOUT_SECONDS = 3
postgres_scheme = urlsplit(POSTGRES_URI).scheme
if postgres_scheme not in {"postgres", "postgresql"}:
    raise ValueError(
        f"{postgres_uri_variable or '本地教学占位符'} 必须是 "
        "postgres:// 或 postgresql:// URI"
    )

connection_source = (
    f"项目 .env / 进程环境中的 {postgres_uri_variable}（值已隐藏）"
    if postgres_uri_variable
    else "本地教学占位符 localhost:5432/langgraph"
)
print("Jupyter 内核当前工作目录：", os.getcwd())
print(
    "项目 .env 加载：",
    "LOADED" if dotenv_loaded else "NOT_FOUND_OR_EMPTY",
)
print("连接来源：", connection_source)
print("安全提示：连接 URI、密码及环境主机地址不会输出。")

postgres_ready = False
postgres_skip_reason = ""
try:
    with psycopg.connect(
        POSTGRES_URI,
        connect_timeout=POSTGRES_CONNECT_TIMEOUT_SECONDS,
    ) as probe_connection:
        postgres_ready = (
            probe_connection.execute("SELECT 1").fetchone() == (1,)
        )
except (psycopg.Error, ValueError) as exc:
    postgres_skip_reason = (
        f"{type(exc).__name__}: PostgreSQL 连接探测失败；"
        "请启动服务并检查 URI、端口、TLS 与认证配置。"
    )

if postgres_ready:
    print("PostgreSQL 探测：READY（SELECT 1 成功）")
else:
    print("PostgreSQL 探测：SKIP")
    print(postgres_skip_reason)


Jupyter 内核当前工作目录： /Users/deltav/Developer/01-Learning-Notes/41-langgraph/langgraph-tutorial/chapter-03-persistence
项目 .env 加载： LOADED
连接来源： 项目 .env / 进程环境中的 LANGCHAIN_POSTGRES_URL（值已隐藏）
安全提示：连接 URI、密码及环境主机地址不会输出。
PostgreSQL 探测：READY（SELECT 1 成功）


## 3. 构造会失败一次的并行图

图的拓扑是 `START → prepare_job → {stable_branch, flaky_branch} → finalize → END`。

- `stable_branch` 与 `flaky_branch` 属于同一 superstep；
- 两个节点都写 `branch_results`，因此该字段必须使用 reducer；
- `add_edge(["stable_branch", "flaky_branch"], "finalize")` 建立显式 join barrier；
- `failure_switch` 模拟外部依赖先不可用、随后恢复；
- `execution_counts` 不属于 State，只用于当前 Notebook 验证节点实际执行次数。

~~~mermaid
flowchart LR
    START --> P[prepare_job]
    P --> S[stable_branch]
    P --> F[flaky_branch]
    S --> J[finalize]
    F --> J
    J --> END
~~~


In [3]:
class RecoveryState(TypedDict, total=False):
    job_id: str
    prepared: str
    branch_results: Annotated[list[str], operator.add]
    report: str


EXPECTED_FAILURE_MESSAGE = "教学用瞬时故障：依赖服务暂不可用"
execution_counts: Counter[str] = Counter()
failure_switch = {"resolved": False}


def prepare_job(state: RecoveryState) -> dict[str, str]:
    """先完成一个能够形成完整 checkpoint 的准备步骤。"""

    execution_counts["prepare_job"] += 1
    return {"prepared": f"prepared:{state['job_id']}"}


def stable_branch(state: RecoveryState) -> dict[str, list[str]]:
    """始终成功；它的结果应在恢复时通过 pending write 复用。"""

    execution_counts["stable_branch"] += 1
    return {"branch_results": [f"stable:{state['job_id']}"]}


def flaky_branch(state: RecoveryState) -> dict[str, list[str]]:
    """第一次运行失败；修复开关后，恢复运行会再次执行本节点。"""

    execution_counts["flaky_branch"] += 1
    if not failure_switch["resolved"]:
        raise RuntimeError(EXPECTED_FAILURE_MESSAGE)
    return {"branch_results": [f"flaky:{state['job_id']}"]}


def finalize(state: RecoveryState) -> dict[str, str]:
    """只有两个并行分支都完成后才会执行。"""

    execution_counts["finalize"] += 1
    return {"report": " + ".join(sorted(state["branch_results"]))}


def build_recovery_graph(checkpointer):
    builder = StateGraph(state_schema=RecoveryState)
    builder.add_node("prepare_job", prepare_job)
    builder.add_node("stable_branch", stable_branch)
    builder.add_node("flaky_branch", flaky_branch)
    builder.add_node("finalize", finalize)
    builder.add_edge(START, "prepare_job")
    builder.add_edge("prepare_job", "stable_branch")
    builder.add_edge("prepare_job", "flaky_branch")
    builder.add_edge(["stable_branch", "flaky_branch"], "finalize")
    builder.add_edge("finalize", END)
    return builder.compile(checkpointer=checkpointer)


RECOVERY_JOB_ID = f"job-{uuid4().hex[:12]}"
RECOVERY_THREAD_ID = f"postgres-recovery-{uuid4().hex}"
recovery_config: RunnableConfig = {
    "configurable": {"thread_id": RECOVERY_THREAD_ID}
}

print("本次教学 job_id：", RECOVERY_JOB_ID)
print("本次教学 thread_id：", RECOVERY_THREAD_ID)


本次教学 job_id： job-ae29e26c2561
本次教学 thread_id： postgres-recovery-13f7eb85363f439b98cf2c72f23cd3e0


## 4. 第一次运行：捕获预期异常并保存失败现场

下面先调用 `setup()`，再以 `durability="sync"` 启动图。代码只捕获消息完全匹配的教学 `RuntimeError`；数据库错误会标为 `SKIP`，断言错误或其他程序缺陷不会被吞掉。

预期执行次数：

| 节点 | 第一次运行后 |
| --- | ---: |
| `prepare_job` | 1 |
| `stable_branch` | 1 |
| `flaky_branch` | 1（失败） |
| `finalize` | 0 |

失败后立即调用 `get_state()`。LangGraph 1.1.2 会把已保存的成功 pending writes 重建到可观察快照中，因此 `values` 已能看到 stable 结果，而 `next` 只保留尚未成功的 `flaky_branch`。


In [4]:
postgres_demo_ready = postgres_ready
failure_captured = False
failed_snapshot = None
failed_checkpoint_id = None
failed_history_count = 0
failed_tasks_by_name = {}
execution_counts.clear()
failure_switch["resolved"] = False

if not postgres_demo_ready:
    print("SKIP：未执行预期失败与 checkpoint 写入。")
else:
    try:
        with PostgresSaver.from_conn_string(POSTGRES_URI) as first_saver:
            # 首次使用目标数据库/schema 时必须调用；重复调用是迁移安全的。
            first_saver.setup()
            first_graph = build_recovery_graph(first_saver)

            try:
                first_graph.invoke(
                    {"job_id": RECOVERY_JOB_ID},
                    recovery_config,
                    durability="sync",
                )
            except RuntimeError as exc:
                if str(exc) != EXPECTED_FAILURE_MESSAGE:
                    raise
                failure_captured = True
            else:
                raise AssertionError("教学故障未触发，无法验证恢复流程")

            failed_snapshot = first_graph.get_state(recovery_config)
            failed_checkpoint_id = failed_snapshot.config[
                "configurable"
            ]["checkpoint_id"]
            failed_history_count = len(
                list(first_graph.get_state_history(recovery_config))
            )
            failed_tasks_by_name = {
                task.name: task for task in failed_snapshot.tasks
            }

            assert execution_counts == Counter(
                {
                    "prepare_job": 1,
                    "stable_branch": 1,
                    "flaky_branch": 1,
                }
            )
            assert failed_snapshot.values["job_id"] == RECOVERY_JOB_ID
            assert failed_snapshot.values["prepared"] == (
                f"prepared:{RECOVERY_JOB_ID}"
            )
            assert failed_snapshot.values["branch_results"] == [
                f"stable:{RECOVERY_JOB_ID}"
            ]
            assert failed_snapshot.next == ("flaky_branch",)
            assert failed_tasks_by_name["stable_branch"].error is None
            assert failed_tasks_by_name["stable_branch"].result == {
                "branch_results": [f"stable:{RECOVERY_JOB_ID}"]
            }
            assert failed_tasks_by_name["flaky_branch"].error is not None
            assert failed_tasks_by_name["flaky_branch"].result is None
            assert failed_history_count == 3
    except psycopg.Error as exc:
        postgres_demo_ready = False
        print("PostgreSQL 第一次运行：SKIP")
        print(
            f"{type(exc).__name__}: setup、checkpoint 或 pending write "
            "写入失败；请检查权限、TLS、连接稳定性与磁盘状态。"
        )
    else:
        print("预期节点失败：VERIFIED")
        print("捕获异常：", EXPECTED_FAILURE_MESSAGE)
        print("第一次运行后的执行次数：", dict(execution_counts))
        print("失败现场 next：", failed_snapshot.next)
        print("失败前完整 checkpoint 数量：", failed_history_count)
        print("第一个 saver 上下文已退出，数据库连接已关闭。")


预期节点失败：VERIFIED
捕获异常： 教学用瞬时故障：依赖服务暂不可用
第一次运行后的执行次数： {'prepare_job': 1, 'stable_branch': 1, 'flaky_branch': 1}
失败现场 next： ('flaky_branch',)
失败前完整 checkpoint 数量： 3
第一个 saver 上下文已退出，数据库连接已关闭。


### 失败现场如何阅读

`StateSnapshot` 的三个部分承担不同职责：

- `values`：从最后完整 checkpoint 加上可复用 pending writes 重建出的当前可观察 State；
- `next`：恢复时仍需执行的任务，本例应只剩 `flaky_branch`；
- `tasks`：本 superstep 的任务详情；成功任务带 `result`，失败任务带 `error`。

因此，“`values` 已有 stable 结果”不等于失败 superstep 已经生成下一份完整 checkpoint。这个结果来自绑定到当前 checkpoint 的 pending write；下一个完整 checkpoint 要等失败任务恢复成功、整个 superstep 完成后才会创建。


In [5]:
if not postgres_demo_ready or not failure_captured:
    print("SKIP：没有可检查的 PostgreSQL 失败现场。")
else:
    print("checkpoint step：", failed_snapshot.metadata["step"])
    print("可观察 values：", dict(failed_snapshot.values))
    print("恢复时待执行 next：", failed_snapshot.next)
    print("任务摘要：")
    for task in failed_snapshot.tasks:
        print(
            {
                "name": task.name,
                "has_result": task.result is not None,
                "has_error": task.error is not None,
            }
        )


checkpoint step： 1
可观察 values： {'job_id': 'job-1a22dde998ed', 'prepared': 'prepared:job-1a22dde998ed', 'branch_results': ['stable:job-1a22dde998ed']}
恢复时待执行 next： ('flaky_branch',)
任务摘要：
{'name': 'stable_branch', 'has_result': True, 'has_error': False}
{'name': 'flaky_branch', 'has_result': False, 'has_error': True}


## 5. PostgreSQL 侧确认 pending writes

应用恢复应优先使用 `get_state()`、`get_state_history()` 和 `invoke(None, ...)`，而不是直接依赖内部表结构。教学上下面额外执行只读 SQL，确认：

- 当前线程已有 3 个完整 checkpoint：input、调度 `prepare_job`、调度并行分支；
- 最新 checkpoint 的 `checkpoint_writes` 中包含 `branch_results`；
- 同一 checkpoint 还包含内部 `__error__` 记录。

查询只读取 channel 名称与行数，不读取序列化 blob，也不打印连接信息。内部表与 channel 名称可能随版本变化，生产业务逻辑不应把它们当作稳定 API。


In [6]:
failed_pending_channels: dict[str, int] = {}
postgres_checkpoint_count_before_resume = 0

if not postgres_demo_ready or not failure_captured:
    print("SKIP：未执行 PostgreSQL pending writes 只读验证。")
else:
    try:
        with psycopg.connect(
            POSTGRES_URI,
            connect_timeout=POSTGRES_CONNECT_TIMEOUT_SECONDS,
        ) as verification_connection:
            channel_rows = verification_connection.execute(
                """
                SELECT channel, count(*)
                FROM checkpoint_writes
                WHERE thread_id = %s
                  AND checkpoint_ns = ''
                  AND checkpoint_id = %s
                GROUP BY channel
                ORDER BY channel
                """,
                (RECOVERY_THREAD_ID, failed_checkpoint_id),
            ).fetchall()
            failed_pending_channels = dict(channel_rows)
            postgres_checkpoint_count_before_resume = (
                verification_connection.execute(
                    """
                    SELECT count(*)
                    FROM checkpoints
                    WHERE thread_id = %s AND checkpoint_ns = ''
                    """,
                    (RECOVERY_THREAD_ID,),
                ).fetchone()[0]
            )

        assert failed_pending_channels.get("branch_results", 0) >= 1
        assert failed_pending_channels.get("__error__", 0) == 1
        assert postgres_checkpoint_count_before_resume == 3
    except psycopg.Error as exc:
        postgres_demo_ready = False
        print("PostgreSQL pending writes 验证：SKIP")
        print(
            f"{type(exc).__name__}: 只读验证失败；"
            "请检查数据库状态、search_path 与读权限。"
        )
    else:
        print("PostgreSQL pending writes：VERIFIED")
        print("最新 checkpoint 的 channel 行数：", failed_pending_channels)
        print(
            "恢复前完整 checkpoint 数量：",
            postgres_checkpoint_count_before_resume,
        )


PostgreSQL pending writes：VERIFIED
最新 checkpoint 的 channel 行数： {'branch_results': 1, '__error__': 1, 'join:stable_branch+flaky_branch:finalize': 1}
恢复前完整 checkpoint 数量： 3


## 6. 新 saver 恢复：`invoke(None, same_config)`

现在把教学故障标记为已修复，然后创建新的 `PostgresSaver` 与已编译图。恢复前先调用 `get_state()`，确认新 saver 从 PostgreSQL 读到了同一个失败 checkpoint；随后用 `None` 继续执行。

恢复后的预期计数是：

| 节点 | 第一次运行后 | 恢复完成后 | 结论 |
| --- | ---: | ---: | --- |
| `prepare_job` | 1 | 1 | 完整 checkpoint 之前的节点不重跑 |
| `stable_branch` | 1 | 1 | 成功 pending write 被复用 |
| `flaky_branch` | 1 | 2 | 失败节点从头重试一次 |
| `finalize` | 0 | 1 | 两个分支齐备后首次执行 |

恢复完成后应共有 5 个完整 checkpoint：失败前 3 个，加上并行 superstep 完成和 `finalize` 完成后的 2 个。


In [7]:
recovered_result = None
recovered_snapshot = None
recovered_history_count = 0

if not postgres_demo_ready or not failure_captured:
    print("SKIP：未执行新 saver 的 checkpoint 恢复。")
else:
    failure_switch["resolved"] = True
    try:
        with PostgresSaver.from_conn_string(POSTGRES_URI) as second_saver:
            second_graph = build_recovery_graph(second_saver)
            persisted_failed_snapshot = second_graph.get_state(
                recovery_config
            )

            assert persisted_failed_snapshot.config["configurable"][
                "checkpoint_id"
            ] == failed_checkpoint_id
            assert persisted_failed_snapshot.next == ("flaky_branch",)
            assert persisted_failed_snapshot.values["branch_results"] == [
                f"stable:{RECOVERY_JOB_ID}"
            ]

            # None 表示继续未完成运行；不要在这里重新传原始输入字典。
            recovered_result = second_graph.invoke(
                None,
                recovery_config,
                durability="sync",
            )
            recovered_snapshot = second_graph.get_state(recovery_config)
            recovered_history_count = len(
                list(second_graph.get_state_history(recovery_config))
            )

            assert execution_counts == Counter(
                {
                    "prepare_job": 1,
                    "stable_branch": 1,
                    "flaky_branch": 2,
                    "finalize": 1,
                }
            )
            assert sorted(recovered_result["branch_results"]) == [
                f"flaky:{RECOVERY_JOB_ID}",
                f"stable:{RECOVERY_JOB_ID}",
            ]
            assert recovered_result["report"] == (
                f"flaky:{RECOVERY_JOB_ID} + stable:{RECOVERY_JOB_ID}"
            )
            assert recovered_snapshot.next == ()
            assert recovered_history_count == 5
    except psycopg.Error as exc:
        postgres_demo_ready = False
        print("PostgreSQL checkpoint 恢复：SKIP")
        print(
            f"{type(exc).__name__}: 读取或恢复期间数据库不可用；"
            "请修复连接后继续使用原 thread_id。"
        )
    else:
        print("PostgreSQL checkpoint 恢复：VERIFIED")
        print("恢复后的结果：", recovered_result)
        print("恢复完成后的执行次数：", dict(execution_counts))
        print("恢复完成后的 next：", recovered_snapshot.next)
        print("恢复完成后的完整 checkpoint 数量：", recovered_history_count)


PostgreSQL checkpoint 恢复：VERIFIED
恢复后的结果： {'job_id': 'job-1a22dde998ed', 'prepared': 'prepared:job-1a22dde998ed', 'branch_results': ['flaky:job-1a22dde998ed', 'stable:job-1a22dde998ed'], 'report': 'flaky:job-1a22dde998ed + stable:job-1a22dde998ed'}
恢复完成后的执行次数： {'prepare_job': 1, 'flaky_branch': 2, 'stable_branch': 1, 'finalize': 1}
恢复完成后的 next： ()
恢复完成后的完整 checkpoint 数量： 5


### 恢复结果的精确含义

- 新 saver 在调用前就能读到失败 checkpoint，证明恢复数据不依赖第一个 saver 对象；
- `prepare_job` 没有重跑，因为它位于已经完成的 superstep 之前；
- `stable_branch` 没有重跑，因为它的 pending write 已绑定到失败现场的 checkpoint；
- `flaky_branch` 从节点函数开头重新执行，因此计数从 1 变成 2；
- 两个分支结果合并后，join barrier 才允许 `finalize` 执行；
- 恢复沿原线程继续追加 checkpoint，不会覆写失败前的历史。

这说明 checkpoint 恢复的粒度既包含 superstep 边界，也包含同一 superstep 内已经完成的任务写入。它不会恢复失败函数的 Python 调用栈或局部变量；失败节点始终从函数入口重新执行。


In [8]:
postgres_checkpoint_count_after_resume = 0
historical_error_rows = 0

if not postgres_demo_ready or recovered_result is None:
    print("SKIP：未执行恢复后的 PostgreSQL 历史验证。")
else:
    try:
        with psycopg.connect(
            POSTGRES_URI,
            connect_timeout=POSTGRES_CONNECT_TIMEOUT_SECONDS,
        ) as verification_connection:
            postgres_checkpoint_count_after_resume = (
                verification_connection.execute(
                    """
                    SELECT count(*)
                    FROM checkpoints
                    WHERE thread_id = %s AND checkpoint_ns = ''
                    """,
                    (RECOVERY_THREAD_ID,),
                ).fetchone()[0]
            )
            historical_error_rows = verification_connection.execute(
                """
                SELECT count(*)
                FROM checkpoint_writes
                WHERE thread_id = %s
                  AND checkpoint_ns = ''
                  AND checkpoint_id = %s
                  AND channel = '__error__'
                """,
                (RECOVERY_THREAD_ID, failed_checkpoint_id),
            ).fetchone()[0]

        assert postgres_checkpoint_count_after_resume == 5
        assert postgres_checkpoint_count_after_resume == recovered_history_count
        assert historical_error_rows == 1
    except psycopg.Error as exc:
        postgres_demo_ready = False
        print("恢复后的 PostgreSQL 历史验证：SKIP")
        print(
            f"{type(exc).__name__}: 只读历史验证失败；"
            "请检查数据库状态与读权限。"
        )
    else:
        print("恢复后的 PostgreSQL 历史：VERIFIED")
        print("完整 checkpoint 数量：", postgres_checkpoint_count_after_resume)
        print("失败 checkpoint 仍保留 error 行：", historical_error_rows)


恢复后的 PostgreSQL 历史：VERIFIED
完整 checkpoint 数量： 5
失败 checkpoint 仍保留 error 行： 1


## 7. 三类“继续运行”不要混淆

| 场景 | 下一次输入 | 含义 |
| --- | --- | --- |
| 未处理异常后继续 | `None` + 相同 `thread_id` | 继续未完成任务，复用成功 pending writes |
| `interrupt()` 后提供人工输入 | `Command(resume=value)` | 把 value 送回中断点；节点会从开头重放到 `interrupt()` |
| 已完成线程开始新一轮业务输入 | 新的 State 输入字典 + 相同 `thread_id` | 从最新 State 开始追加一轮新执行 |

若异常仍未修复，`invoke(None, config)` 会再次执行失败节点并再次抛错。PostgresSaver 负责保存执行状态，不会自动修复网络、认证、限流、数据质量或业务逻辑问题。

自动重试与人工/运维恢复可以组合：可为节点配置 `RetryPolicy` 处理短暂抖动；重试耗尽后让异常冒泡，待外部问题解决，再从 checkpoint 恢复。本项目的 LangGraph 1.1.2 中，`RetryPolicy.max_attempts` 包含首次调用。


## 8. 生产边界与最佳实践

### 外部副作用必须幂等

checkpoint 能避免重新执行“已经成功并保存 pending writes”的任务，但不能把任意外部系统操作变成 exactly-once。失败节点可能已经扣款、发邮件或写入第三方系统，随后才在返回 State 更新之前抛错；恢复时它会从函数开头再次运行。

常见做法：

- 使用稳定的业务幂等键，例如 `job_id + node_name`；
- 在外部系统写入前查询是否已经完成；
- 使用数据库唯一约束、事务或 outbox/inbox 模式；
- 把大节点拆成边界清晰的小节点，缩小失败后需要重做的范围。

### 其他边界

- **硬崩溃窗口**：进程被强制终止时，最近一次异步写入可能尚未完成。关键流程优先使用 `durability="sync"`，但仍应按 at-least-once 思维设计副作用；
- **图代码兼容性**：恢复使用当前部署的图代码。不要在仍有未完成线程时直接删除/重命名待执行节点，State Schema 与 reducer 变更也要兼容旧 checkpoint；
- **并发恢复**：避免多个 worker 同时恢复同一个业务线程；在调度层使用租约、队列去重或业务锁；
- **数据库可靠性**：配置 TLS、连接池、超时、备份、监控和保留策略；checkpoint 可靠性不会超过 PostgreSQL 部署本身；
- **序列化与安全**：State 只保存恢复所需数据，避免不可信复杂对象与不必要的敏感内容，并限制数据库访问权限；
- **可观测性**：记录 `thread_id`、业务幂等键、失败节点和恢复次数，但不要记录数据库 URI、令牌或敏感 State。


## 9. 可选清理：只删除本次教学线程

默认保留本次唯一线程，便于关闭 Notebook 后继续检查。若确认不再需要，把 `CLEAN_UP_RECOVERY_THREAD` 改为 `True` 后单独运行下一单元。

`delete_thread()` 会删除该线程的 checkpoints、blobs 与 writes，属于不可逆的数据删除操作，因此示例默认不执行。它只针对本 Notebook 随机生成的 `RECOVERY_THREAD_ID`，不会清理其他线程。


In [9]:
CLEAN_UP_RECOVERY_THREAD = False

if not postgres_ready:
    print("SKIP：PostgreSQL 不可用，没有执行清理。")
elif not CLEAN_UP_RECOVERY_THREAD:
    print("保留本次教学线程；未执行 delete_thread()。")
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as cleanup_saver:
        cleanup_saver.delete_thread(RECOVERY_THREAD_ID)
    print("已删除本 Notebook 生成的教学线程。")


保留本次教学线程；未执行 delete_thread()。


## 10. 验收标记与排障

数据库可用且全部断言通过时，应依次看到：

1. `PostgreSQL 探测：READY`；
2. `预期节点失败：VERIFIED`；
3. `PostgreSQL pending writes：VERIFIED`；
4. `PostgreSQL checkpoint 恢复：VERIFIED`；
5. `恢复后的 PostgreSQL 历史：VERIFIED`。

若看到 `SKIP`，请先运行 `02-postgres-saver.ipynb` 的前置条件与连通性检查。常见原因包括 PostgreSQL 未启动、URI/TLS/认证错误、首次迁移权限不足或 search path 不一致。不要把“Notebook 代码执行完且数据库步骤被跳过”描述成“Postgres 恢复已验证”。

### 官方资料

- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangGraph Checkpointing API Reference](https://reference.langchain.com/python/langgraph/checkpoints)
- [PostgresSaver API Reference](https://reference.langchain.com/python/langgraph.checkpoint.postgres/PostgresSaver)
- [LangGraph Checkpoint README：Pending writes](https://github.com/langchain-ai/langgraph/blob/main/libs/checkpoint/README.md#pending-writes)

> 官方网站会随新版本更新。本笔记中的执行断言以项目锁定的 LangGraph 1.1.2 与本地已安装源码为准，不直接套用只在 LangGraph 1.2+ 提供的 node timeout、error handler 等新接口。
